# Extended functional specialization (5 per schedule, 8 runs)

Runs the specialization pipeline (retraining, correlation, ablations) on **5 participants per schedule** (same, near, far) for each of **8 run configurations**, then presents individual and averaged results in tables and figures sorted by sparsity and input scheme.

**Run configurations:** No comms (2), sparsity 0.3 (2), 0.5 (2), 0.7 (2), full (2) — each with shared and task_routed input.

## 1. Setup and imports

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import ConcatDataset, DataLoader
from tqdm.auto import tqdm

project_root = Path(os.getcwd()).resolve()
while not (project_root / "a1b2").exists():
    if project_root == project_root.parent:
        raise RuntimeError("Project root (containing a1b2) not found.")
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

data_folder = project_root / "data"
sim_folder = data_folder / "simulations"

from a1b2.analysis import transfer_interference as ann
from a1b2.analysis import run_loader
from a1b2.analysis.retraining_a1b2 import (
    create_retraining_model_a1b2,
    train_probe_readout_a1b2,
    eval_probe_readout_a1b2,
    retraining_specialization_scalar,
    compute_ablations_metric_a1b2,
)
from a1b2.analysis.correlations_a1b2 import compute_correlation_metric_a1b2
from a1b2.data.basic_funcs import get_datasets
from a1b2.models.ffn import CreateParticipantDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
project_root, data_folder, sim_folder, device

/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(PosixPath('/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular'),
 PosixPath('/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data'),
 PosixPath('/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations'),
 device(type='cuda'))

## 2. Run configuration

Define the 8 run prefixes and canonical ordering (sparsity then input_scheme) for sorting and plotting.

In [2]:
# (prefix, sparsity_label, input_scheme) in display order: no_comms -> 0.3 -> 0.5 -> 0.7 -> full; within each, shared then task_routed
RUN_CONFIG = [
    ("two_module_rnn_25_no_comms_nb2", "no_comms", "shared"),
    ("two_module_rnn_25_task_routed_no_comms_nb2", "no_comms", "task_routed"),
    ("two_module_rnn_25_low_sparse_nb2", "0.3", "shared"),
    ("two_module_rnn_25_task_routed_low_sparse_nb2", "0.3", "task_routed"),
    ("two_module_rnn_25_sp05_nb2", "0.5", "shared"),
    ("two_module_rnn_25_task_routed_sp05_nb2", "0.5", "task_routed"),
    ("two_module_rnn_25_sp07_nb2", "0.7", "shared"),
    ("two_module_rnn_25_task_routed_sp07_nb2", "0.7", "task_routed"),
    ("two_module_rnn_25_nb2", "full", "shared"),
    ("two_module_rnn_25_task_routed_nb2", "full", "task_routed"),
]

def resolve_run(prefix):
    if not sim_folder.exists():
        raise FileNotFoundError(f"Simulations folder not found: {sim_folder}")
    candidates = sorted([p for p in sim_folder.iterdir() if p.is_dir() and p.name.startswith(prefix)])
    if not candidates:
        return None
    return candidates[0]

runs = []
for prefix, sparsity_label, input_scheme in RUN_CONFIG:
    path = resolve_run(prefix)
    if path is not None:
        runs.append({"run_id": prefix, "path": path, "sparsity_label": sparsity_label, "input_scheme": input_scheme})
    else:
        print(f"Skip {prefix}: no matching folder.")
runs

[{'run_id': 'two_module_rnn_25_no_comms_nb2',
  'path': PosixPath('/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations/two_module_rnn_25_no_comms_nb2_nb2_shared_sp0_sep_cr_RNN'),
  'sparsity_label': 'no_comms',
  'input_scheme': 'shared'},
 {'run_id': 'two_module_rnn_25_task_routed_no_comms_nb2',
  'path': PosixPath('/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations/two_module_rnn_25_task_routed_no_comms_nb2_nb2_task_routed_sp0_sep_cr_RNN'),
  'sparsity_label': 'no_comms',
  'input_scheme': 'task_routed'},
 {'run_id': 'two_module_rnn_25_low_sparse_nb2',
  'path': PosixPath('/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations/two_module_rnn_25_low_sparse_nb2_nb2_shared_sp0.3_sep_cr_RNN'),
  'sparsity_label': '0.3',
  'input_scheme': 'shared'},
 {'run_id': 'two_module_rnn_25_task_routed_low_sparse_nb2',
  'path': PosixPath('/home/kat/workspace/

## 3. Participant selection (5 per schedule)

For each run: get participants (state+npz or npz-only), parse schedule from id, take first 5 per schedule (same, near, far).

In [3]:
def participant_id_to_schedule(pid):
    pid = str(pid).lower()
    if "same" in pid:
        return "same"
    if "near" in pid:
        return "near"
    if "far" in pid:
        return "far"
    return None

def get_tasks_for_run(r):
    """Return list of (run_dict, participant, schedule) for one run (5 per schedule)."""
    run_path = r["path"]
    candidates = run_loader.list_participants_with_state(run_path)
    if not candidates:
        candidates = run_loader.list_participants_with_npz(run_path)
    candidates = sorted([p for p in candidates if p in participants_in_trial])
    by_schedule = {"same": [], "near": [], "far": []}
    for p in candidates:
        s = participant_id_to_schedule(p)
        if s and len(by_schedule[s]) < N_PER_SCHEDULE:
            by_schedule[s].append(p)
    out = []
    for schedule in ["same", "near", "far"]:
        for participant in by_schedule[schedule]:
            out.append((r, participant, schedule))
    return out

N_PER_SCHEDULE = 5
df_trial = ann.load_participant_data(str(data_folder))
participants_in_trial = set(df_trial["participant"].unique())

tasks = []
tasks_by_run = {}  # run_id -> list of (run_dict, participant, schedule)
for r in runs:
    run_path = r["path"]
    candidates = run_loader.list_participants_with_state(run_path)
    if not candidates:
        candidates = run_loader.list_participants_with_npz(run_path)
    candidates = sorted([p for p in candidates if p in participants_in_trial])
    by_schedule = {"same": [], "near": [], "far": []}
    for p in candidates:
        s = participant_id_to_schedule(p)
        if s and len(by_schedule[s]) < N_PER_SCHEDULE:
            by_schedule[s].append(p)
    this_run_tasks = []
    for schedule in ["same", "near", "far"]:
        for participant in by_schedule[schedule]:
            t = (r, participant, schedule)
            tasks.append(t)
            this_run_tasks.append(t)
    tasks_by_run[r["run_id"]] = this_run_tasks

print(f"Total tasks: {len(tasks)} (runs × participants)")
print(f"Participants per run: {len(tasks) // len(runs) if runs else 0}")

Total tasks: 150 (runs × participants)
Participants per run: 15


## 4. Helper: build loader

In [4]:
def build_loader_for_participant(df, participant, task_parameters, batch_size=32, shuffle=False):
    dataset_A1, dataset_B, dataset_A2, _, _ = get_datasets(df, participant, task_parameters)
    combined = ConcatDataset([
        CreateParticipantDataset(dataset_A1),
        CreateParticipantDataset(dataset_B),
        CreateParticipantDataset(dataset_A2),
    ])
    return DataLoader(combined, batch_size=batch_size, shuffle=shuffle)

## 5. Compute metrics (per configuration)

Run **one cell per configuration** so you can run only a subset (e.g. no_comms first). Re-running a configuration cell replaces that run's results. After running the configs you want, run **Build df_results** below, then Tables and Figures.

In [5]:
# Run once to start; re-run to clear all results before re-running configs.
results_list = []
n_epochs_probe = 1

def run_metrics_for_config(run_idx):
    """Run specialization metrics for one run (up to 15 participants). Re-running replaces that run's results."""
    if run_idx < 0 or run_idx >= len(runs):
        print(f"run_idx must be 0..{len(runs)-1}")
        return
    r = runs[run_idx]
    run_id = r["run_id"]
    tasks_this_run = tasks_by_run.get(run_id, [])
    if not tasks_this_run:
        print(f"Run {run_idx} ({run_id}): no participants, skipping.")
        return
    # Replace any existing results for this run
    results_list[:] = [row for row in results_list if row["run_id"] != run_id]
    for r, participant, schedule in tqdm(tasks_this_run, desc=f"Run {run_idx} {run_id}"):
        run_path = r["path"]
        sparsity_label = r["sparsity_label"]
        input_scheme = r["input_scheme"]
        settings = run_loader.load_settings(run_path)
        task_parameters = settings.get("task_parameters") or ann.setup_task_parameters()
        row = {
            "run_id": run_id,
            "participant": participant,
            "schedule": schedule,
            "sparsity_label": sparsity_label,
            "input_scheme": input_scheme,
            "retraining_specialization": None,
            "correlation_specialization": None,
            "ablation_specialization": None,
        }
        has_state = (run_path / f"state_{participant}.pt").exists()
        if has_state:
            loader = build_loader_for_participant(df_trial, participant, task_parameters, batch_size=32, shuffle=True)
            wrapper = run_loader.build_wrapper_from_settings(settings, device=device)
            run_loader.load_wrapper_state(wrapper, run_path / f"state_{participant}.pt")
            wrapper = create_retraining_model_a1b2(wrapper, device=device)
            wrapper = train_probe_readout_a1b2(wrapper, loader, settings.get("condition", {}), n_epochs=n_epochs_probe, lr=1e-3, device=device)
            acc = eval_probe_readout_a1b2(wrapper, loader, device=device)
            row["retraining_specialization"] = float(retraining_specialization_scalar(acc[0, 0], acc[1, 0], acc[0, 1], acc[1, 1]))
            ab = compute_ablations_metric_a1b2(wrapper, loader, device=device)
            row["ablation_specialization"] = float(ab["ablation_specialization"])
        npz_path = run_path / f"sim_{participant}.npz"
        if npz_path.exists():
            with np.load(npz_path, allow_pickle=True) as data:
                if "hiddens_per_module" in data and "probes" in data:
                    participant_data = {"hiddens_per_module": data["hiddens_per_module"], "probes": data["probes"], "inputs": data["inputs"]}
                    out = compute_correlation_metric_a1b2(participant_data, n_samples=5)
                    row["correlation_specialization"] = float(out["correlation_specialization"])
        results_list.append(row)
    print(f"Run {run_idx} ({run_id}): {len(tasks_this_run)} participants appended. Total results: {len(results_list)}")

In [6]:
# Optional: clear all results and start over (then re-run only the config cells you want).
results_list = []
print("results_list cleared.")

results_list cleared.


In [ ]:
run_metrics_for_config(0)  # no_comms shared

Run 0 two_module_rnn_25_no_comms_nb2:   0%|          | 0/15 [00:00<?, ?it/s]

In [ ]:
run_metrics_for_config(1)  # no_comms task_routed

In [ ]:
run_metrics_for_config(2)  # 0.3 shared

In [ ]:
run_metrics_for_config(3)  # 0.3 task_routed

In [ ]:
run_metrics_for_config(4)  # 0.5 shared

In [ ]:
run_metrics_for_config(5)  # 0.5 task_routed

In [ ]:
run_metrics_for_config(6)  # 0.7 shared

In [ ]:
run_metrics_for_config(7)  # 0.7 task_routed

In [ ]:
run_metrics_for_config(8)  # full shared

In [ ]:
run_metrics_for_config(9)  # full task_routed

### Build DataFrame

Run this after one or more config cells. Tables and Figures below use `df_results`.

df_results = pd.DataFrame(results_list).drop_duplicates(subset=["run_id", "participant"], keep="last")
print(f"df_results: {len(df_results)} rows.")

In [ ]:
## 6. Tables

Individual, run-level mean (SEM), and by-schedule mean (SEM), sorted by sparsity then input_scheme.

In [ ]:
metric_cols = ["retraining_specialization", "correlation_specialization", "ablation_specialization"]
sparsity_order = {"no_comms": 0, "0.3": 1, "0.5": 2, "0.7": 3, "full": 4}
if len(df_results) == 0:
    print("No results yet. Run at least one 'Run metrics' config cell (and the init cell), then Build DataFrame.")
    df_sorted = pd.DataFrame()
    run_level = pd.DataFrame()
    by_schedule = pd.DataFrame()
else:
    df_results["sparsity_order"] = df_results["sparsity_label"].map(sparsity_order)
    df_sorted = df_results.sort_values(["sparsity_order", "input_scheme", "run_id", "schedule", "participant"]).drop(columns=["sparsity_order"])

    print("Individual results (first rows):")
    display(df_sorted.head(20))

    run_level = df_sorted.groupby(["run_id", "sparsity_label", "input_scheme"], sort=False)[metric_cols].agg(["mean", "sem", "count"]).reset_index()
    run_level["_so"] = run_level["sparsity_label"].map(sparsity_order)
    run_level = run_level.sort_values(["_so", "input_scheme"]).drop(columns=["_so"])
    print("\nRun-level mean (SEM), n:")
    display(run_level)

    by_schedule = df_sorted.groupby(["run_id", "sparsity_label", "input_scheme", "schedule"], sort=False)[metric_cols].agg(["mean", "sem", "count"]).reset_index()
    by_schedule["_so"] = by_schedule["sparsity_label"].map(sparsity_order)
    by_schedule = by_schedule.sort_values(["_so", "input_scheme", "schedule"]).drop(columns=["_so"])
    print("\nBy run_id and schedule (mean, SEM, n):")
    display(by_schedule)

## 7. Figures

Run-level: mean specialization (± SEM) per run, grouped by sparsity and input_scheme. By task similarity: same with schedule as color/facet.

In [ ]:
import matplotlib.pyplot as plt

if len(df_sorted) == 0:
    print("No results to plot. Run config cells and Build DataFrame first.")
else:
    run_level_plot = df_sorted.groupby(["run_id", "sparsity_label", "input_scheme"], sort=False)[metric_cols].agg(["mean", "sem"]).reset_index()
    run_level_plot["sparsity_order"] = run_level_plot["sparsity_label"].map(sparsity_order)
    run_level_plot = run_level_plot.sort_values(["sparsity_order", "input_scheme"])
    run_level_plot["x_label"] = run_level_plot["sparsity_label"] + "\n" + run_level_plot["input_scheme"]
    x_pos = np.arange(len(run_level_plot))

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for i, m in enumerate(metric_cols):
        ax = axes[i]
        means = run_level_plot[m]["mean"].values
        sems = run_level_plot[m]["sem"].values
        ax.bar(x_pos, means, yerr=sems, capsize=2)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(run_level_plot["x_label"], rotation=45, ha="right")
        ax.set_ylabel("Specialization")
        ax.set_title(m.replace("_", " ").title())
        ax.set_ylim(0, None)
    plt.suptitle("Run-level mean specialization (SEM)")
    plt.tight_layout()
    plt.show()

In [ ]:
if len(df_sorted) == 0:
    print("No results to plot.")
else:
    _rlp = globals().get("run_level_plot")
    if _rlp is None or (hasattr(_rlp, "__len__") and len(_rlp) == 0):
        run_level_plot = df_sorted.groupby(["run_id", "sparsity_label", "input_scheme"], sort=False)[metric_cols].agg(["mean", "sem"]).reset_index()
        run_level_plot["sparsity_order"] = run_level_plot["sparsity_label"].map(sparsity_order)
        run_level_plot = run_level_plot.sort_values(["sparsity_order", "input_scheme"])
        run_level_plot["x_label"] = run_level_plot["sparsity_label"] + "\n" + run_level_plot["input_scheme"]
    fig2, axes2 = plt.subplots(1, 3, figsize=(14, 4))
    schedules = ["same", "near", "far"]
    x = np.arange(len(run_level_plot))
    w = 0.25
    run_keys = run_level_plot[["run_id", "sparsity_label", "input_scheme"]]
    for i, m in enumerate(metric_cols):
        ax = axes2[i]
        for j, sched in enumerate(schedules):
            sub = by_schedule[by_schedule["schedule"] == sched]
            merged = run_keys.merge(sub, on=["run_id", "sparsity_label", "input_scheme"], how="left")
            means = merged[(m, "mean")].values
            sems = merged[(m, "sem")].values
            off = (j - 1) * w
            ax.bar(x + off, means, w, yerr=sems, label=sched, capsize=1)
        ax.set_xticks(x)
        ax.set_xticklabels(run_level_plot["x_label"], rotation=45, ha="right")
        ax.set_ylabel("Specialization")
        ax.set_title(m.replace("_", " ").title())
        ax.legend()
        ax.set_ylim(0, None)
    plt.suptitle("Mean specialization by schedule (same / near / far)")
    plt.tight_layout()
    plt.show()